In [ ]:
import sys 

""" Run this cell before you run anything. This ensures the modular imports work correctly"""

working_root_directory="/teamspace/studios/this_studio/resnet18_cifar10_classifier"

if working_root_directory not in sys.path: 
    sys.path.append(working_root_directory)

In [ ]:
from utils.helper_utils import (
    load_prepare_cifar10,
    build_transforms
    )


from src.test import load_model_for_test,test_model
from utils.helper_utils import build_dataloader,measure_inference_mb_ms

from config.exp_config import dataset_dir,csv_file_path
from typing import List
from copy import deepcopy
import torch

In [92]:
#Device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [93]:
def get_datasets(add_standard_augmentation:bool,add_random_erasing:bool):
        #download cifar10 dataset if it doesn't exist or it is corrupted
        train_dataset,val_dataset,test_dataset=load_prepare_cifar10(
                dataset_path=f"{dataset_dir}", #notice i have chaged the dataset dir due to current script running jupyter notebook dir
                build_transforms=build_transforms,
                image_size=224,
                standard_aug=add_standard_augmentation,#set False not to be applied standard augmentation or True
                add_random_erasing=add_random_erasing
                )
        return train_dataset,val_dataset,test_dataset 

# Get dataset (Applied standard augmentation,Add random erasing)

In [153]:
train_dataset,val_dataset,test_dataset=get_datasets(add_standard_augmentation=True,add_random_erasing=True)

downloaded cifar10 dataset ...


## Prepare the data

# Prepare DataLoader with Cutmix/Mixup Application

In [154]:
train_loader,val_loader,test_loader=build_dataloader(
    batch_size=128,
    num_workers=4,
    use_cutmix_mixup=True,
    device=device,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset
    )


## Load exp5 version 3.0 best candidate model

In [158]:
original_exp5_v3_model=load_model_for_test(
    is_baseline=False,
    model_dir=csv_file_path,
    experiment_name="/kaggle/input/datasets/zarniminnnyan/exp5-v3/exp5_v3/exp5_v3",
    num_classes=10,
    device=device
    )

change the head of resnet18 to 10
--- Extracting Configurations ---
Extracted learning rates from training: [5.8843356115025935e-05, 8.791036923815453e-05, 0.000872719723562627]


In [159]:
exp5_v3_history=test_model(
    model=original_exp5_v3_model,
    test_dataloader=test_loader,
    num_classes=10,
    device=device,
    test_result_dir="./result/exp5_v3"
        )

test model is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

In [160]:
print(f"Exp 5 V 3.0 original test accuracy: {exp5_v3_history["test_acc"]}")

Exp 5 V 3.0 original test accuracy: 94.31


In [161]:
def measure_tensor_storage(model):
    tensors = {}

    # Count parameters and buffers without double-counting shared tensors
    for tensor in list(model.parameters()) + list(model.buffers()):
        tensors[tensor.data_ptr()] = tensor

    total_bytes = sum(
        tensor.numel() * tensor.element_size()
        for tensor in tensors.values()
    )

    return total_bytes / (1024 ** 2)


# Original Experiment 5 inference time and size on CPU

In [163]:
results=measure_inference_mb_ms(model=original_exp5_v3_model.model,input_shape=(1,3,224,224),save_path="../original_v3_inference",experiment_name="exp5_v3",device=torch.device("cpu"))
print(f"Exp 5 V 3.0 size and inference time before Quantization: {results}")

Average latency: 32.830 ms
State dict size: 44.80 MB
Exp 5 V 3.0 size and inference time before Quantization: {'model_time_ms': 32.82951146999039, 'state_dict_size_mb': 44.803531, 'exported_file_size_mb': None}


In [164]:
# model for static quantization
static_model_to_quantize=deepcopy(original_exp5_v3_model.model)
# model for qat quantization 
qat_model_to_quantize=deepcopy(original_exp5_v3_model.model)

## Static PTQ Quantization 

In [165]:
import torch
import torchvision.models as models
import copy
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from torchao.quantization.pt2e import move_exported_model_to_train,move_exported_model_to_eval
import torchao.quantization.pt2e.quantizer.x86_inductor_quantizer as xiq
from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import X86InductorQuantizer
from torch.export import export
from torch.export.dynamic_shapes import Dim


def export_model_for_quantize(model):
    # Set the model to eval mode (you don't need to reset to eval mode at the calibration step)
    model = model.eval()

    # Define the first dimension (batch size) as dynamic
    batch_dim = Dim("batch")
    dynamic_shapes = {"x": {0: batch_dim}}
    
    
    # Create the data, using the dummy data here as an example
    traced_bs =128
    # Random input tensor in channels-last format (NHWC), optimized for convs on GPU and supported on CPU
    x = torch.randn(traced_bs, 3, 224, 224).contiguous(memory_format=torch.channels_last)
    example_inputs = (x,)
    
    # Capture the FX Graph to be quantized
    with torch.no_grad():
        # Note: requires torch >= 2.6 (using export is recomended in pytorch document)
        exported_model = export(
            model,
            args=example_inputs,
            dynamic_shapes=dynamic_shapes,
            strict=True
        ).module()
    return exported_model


## Apply configuration for Quantization targeting CPU (x86)

In [166]:
quantizer = X86InductorQuantizer()
quantizer.set_global(xiq.get_default_x86_inductor_quantization_config())

In [167]:
#exported model for static PTQ 
exported_model=export_model_for_quantize(static_model_to_quantize)

In [168]:
# prepare the model for Quantization-ready (Fuse BatchNorm + Conv2d and insert obversers at the appropriate places)
prepared_model = prepare_pt2e(exported_model, quantizer)

# Calibrate on GPU and Move back to CPU after quantized for compilation and CPU target

In [169]:


# define the dataset to calibrate
def calibrate(model:nn.Module, data_loader:torch.utils.data.DataLoader,requested_batches:int,device:torch.device):
    import tqdm.auto as tqdm

    
    model=move_exported_model_to_eval(model) #move to eval model before calibration (important for compile)
    total_batches=len(data_loader) 
    requested_loader_batches=min(requested_batches,total_batches)
    model.to(device)
    with torch.no_grad():
        for idx,(image,_) in enumerate(tqdm.tqdm(data_loader)):
            # for compatability with the torch.export 
            image = image.contiguous(memory_format=torch.channels_last).to(device)
            model(image)
            if idx+1>= requested_loader_batches: 
                print(f"Requested {requested_loader_batches} batches has reached")
                break 
        print(f"Calibration has finished after processing {requested_loader_batches} out of {total_batches}")
    return model

In [170]:
#calibrate for static PTQ
calibrated_model=calibrate(prepared_model,val_loader,50,device) 
#convert to pt2e quantized model
quantized_model = convert_pt2e(calibrated_model)

  0%|          | 0/79 [00:00<?, ?it/s]

Requested 50 batches has reached
Calibration has finished after processing 50 out of 79


In [171]:
#moved quantized model back to CPU for CPU inference target 
quantized_model=quantized_model.to("cpu")

# Check the reliablity of Quantization helper functions

In [172]:
def check_if_quantized(quantized_model):
    
    conv_nodes = [n for n in quantized_model.graph.nodes if n.op == "call_function" and "conv2d" in str(n.target)]
    linear_nodes = [n for n in quantized_model.graph.nodes if n.op == "call_function" and "linear" in str(n.target)]
    
    def has_dq_input(node):
        return any("dequantize" in str(a.target) for a in node.args if hasattr(a, "target"))

    q_conv = sum(1 for n in conv_nodes if has_dq_input(n))
    q_linear = sum(1 for n in linear_nodes if has_dq_input(n))
    
    print(f"Conv2d:  {q_conv}/{len(conv_nodes)} quantized")
    print(f"Linear:  {q_linear}/{len(linear_nodes)} quantized")
    print(f"Total:   {q_conv + q_linear}/{len(conv_nodes) + len(linear_nodes)} quantized")
    
    if q_conv + q_linear == 0:
        print(">>> BUG: nothing quantized!")
    elif q_conv + q_linear < len(conv_nodes) + len(linear_nodes):
        print(">>> PARTIAL: some ops not quantized")
    else:
        print(">>> ALL ops quantized")


In [173]:
check_if_quantized(quantized_model)

Conv2d:  20/20 quantized
Linear:  1/1 quantized
Total:   21/21 quantized
>>> ALL ops quantized


In [174]:
def check_weights_type(quantized_model):
    from collections import Counter
    dtype_counts = Counter(
    str(t.dtype)
    for t in list(quantized_model.parameters())
    + list(quantized_model.buffers())
    )
    print(dtype_counts)

In [175]:
check_weights_type(quantized_model)


Counter({'torch.float32': 63, 'torch.int64': 21, 'torch.int8': 21})


In [176]:
def check_each_weight_type_memory_usage(quantized_model):
    
    for name, tensor in quantized_model.state_dict().items():
        size_mb = tensor.numel() * tensor.element_size() / (1024 ** 2)

        print(
            f"{name:70s} "
            f"{str(tensor.dtype):15s} "
            f"{str(tuple(tensor.shape)):25s} "
            f"{size_mb:.3f} MiB"
        )

In [177]:
def check_unused_weights(quantized_model):
    # Get attributes referenced by the graph
    used_attrs = {
        str(node.target)
        for node in quantized_model.graph.nodes
        if node.op == "get_attr"
    }
    
    # Find FP32 state tensors that are not directly referenced
    for name, tensor in quantized_model.state_dict().items():
        if tensor.dtype == torch.float32 and name not in used_attrs:
            print(name, tuple(tensor.shape))

In [145]:
check_unused_weights(quantized_model)

conv1.weight (64, 3, 7, 7)
layer1.0.conv1.weight (64, 64, 3, 3)
layer1.0.conv2.weight (64, 64, 3, 3)
layer1.1.conv1.weight (64, 64, 3, 3)
layer1.1.conv2.weight (64, 64, 3, 3)
layer2.0.conv1.weight (128, 64, 3, 3)
layer2.0.conv2.weight (128, 128, 3, 3)
layer2.0.downsample.0.weight (128, 64, 1, 1)
layer2.1.conv1.weight (128, 128, 3, 3)
layer2.1.conv2.weight (128, 128, 3, 3)
layer3.0.conv1.weight (256, 128, 3, 3)
layer3.0.conv2.weight (256, 256, 3, 3)
layer3.0.downsample.0.weight (256, 128, 1, 1)
layer3.1.conv1.weight (256, 256, 3, 3)
layer3.1.conv2.weight (256, 256, 3, 3)
layer4.0.conv1.weight (512, 256, 3, 3)
layer4.0.conv2.weight (512, 512, 3, 3)
layer4.0.downsample.0.weight (512, 256, 1, 1)
layer4.1.conv1.weight (512, 512, 3, 3)
layer4.1.conv2.weight (512, 512, 3, 3)
fc.weight (10, 512)


In [178]:

def rebuild_clean_graph_and_save(quantized_model,filename):
    
    import torch.fx as fx

    # Define the first dimension (batch size) as dynamic
    batch_dim = Dim("batch")
    dynamic_shapes = {"x": {0: batch_dim}}
    
    
    # Create the data, using the dummy data here as an example
    traced_bs =128
    # Random input tensor in channels-last format (NHWC), optimized for convs on GPU and supported on CPU
    x = torch.randn(traced_bs, 3, 224, 224).contiguous(memory_format=torch.channels_last)
    example_inputs = (x,)
    
    
    # Rebuild using only attributes referenced by the graph
    clean_model = fx.GraphModule(
    quantized_model,
    quantized_model.graph
)
    clean_model.eval()

    # Verify that outputs remain equivalent
    with torch.no_grad():
        y_original = quantized_model(*example_inputs)
        y_clean = clean_model(*example_inputs)
    
    torch.testing.assert_close(
        y_clean, y_original,
        rtol=1e-4, atol=1e-5
    )

    clean_exported = torch.export.export(
    clean_model,
    example_inputs,
    dynamic_shapes=dynamic_shapes
    )
    
    torch.export.save(
    clean_exported,
    f"../{filename}.pt2e"
)
    
    print("Output verification passed.")
    print("Original state entries:", len(quantized_model.state_dict()))
    print("Clean state entries:", len(clean_model.state_dict()))

    #convert it to module with .module() to remove FX wrapper
    
    return clean_exported.module()

In [179]:
clean_exported_static=rebuild_clean_graph_and_save(quantized_model,"last_static_ptq_model")


W1018 08:03:51.364000 58 torch/export/pt2_archive/_package.py:686] Expect archive file to be a file ending in .pt2, or is a buffer. Instead got {../last_static_ptq_model.pt2e}


Output verification passed.
Original state entries: 105
Clean state entries: 84


# Compile static model to use torch Inductor C++ wrapper for faster inference

In [180]:
def torch_inductor(quantized_model,traced_bs):
    # using the C++ wrapper instead of default Python wrapper
    import torch._inductor.config as config
    config.cpp_wrapper = True
    #Enable advanced CPU vectorization and kernel generation
    config.cpp.enable_kernel_profile = False
    # Force inductor to generate optimized int8 loop bodies
    config.freezing = True

    x = torch.randn(traced_bs, 3, 224, 224).contiguous(memory_format=torch.channels_last)
    example_inputs = (x,)
    with torch.no_grad():
        optimized_model = torch.compile(quantized_model,mode="max-autotune")
        # Running some benchmark
        print("Compiling and tuning kernels... (This takes a moment)")
        optimized_model(*example_inputs)
        print("Compilation successful!")    

    return optimized_model

In [181]:
compiled_static_model=torch_inductor(clean_exported_static,128)

Compiling and tuning kernels... (This takes a moment)


skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()


Compilation successful!


In [182]:
static_exp5_v3_history=test_model(
    model=compiled_static_model,
    test_dataloader=test_loader,
    num_classes=10,
    device=torch.device("cpu"),
    test_result_dir="./result/exp5_v3"
    
        )

test model is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()
skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()


In [183]:
print(static_exp5_v3_history["test_acc"])

91.82


In [184]:
results=measure_inference_mb_ms(model=compiled_static_model,input_shape=(1,3,224,224),save_path="../last_static_v3_inference",experiment_name="exp5_v3",device=torch.device("cpu"))
print(f"Exp 5 V 3.0 size and inference time after Quantization: {results}")

skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()


Average latency: 12.914 ms
State dict size: 11.28 MB
Exp 5 V 3.0 size and inference time after Quantization: {'model_time_ms': 12.914471259982747, 'state_dict_size_mb': 11.276007, 'exported_file_size_mb': None}


# QAT quantization to preserve the better accuracy

In [185]:
import torch
import torchao
import tqdm.auto as tqdm
from torchao.quantization.pt2e.quantize_pt2e import (
  prepare_qat_pt2e,
  convert_pt2e,
)
import torchao.quantization.pt2e.quantizer.x86_inductor_quantizer as xiq
from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import X86InductorQuantizer

def qat_evaluate(model,device,val_loader):
    """
    Evaluates the model on val_loader and prints accuracy.
    """
    
    try:
        model.eval()
    except NotImplementedError:
        model = torchao.quantization.pt2e.move_exported_model_to_eval(model)

    model.to(device)
    correct = total = 0
    with torch.no_grad():
        validation_loader=tqdm.tqdm(val_loader,desc=f"Validation is in progress")
        for x, y in validation_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    accuracy = correct / total
    return accuracy



def qat_train(model, loader,val_loader, epochs,device,lr):
    """
    Trains a model with SGD and cross-entropy loss.
    Loads from save_path if it exists.
    """

    try:
        model.train()
    except NotImplementedError:
        torchao.quantization.pt2e.move_exported_model_to_train(model)
   
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        loader=tqdm.tqdm(loader,desc=f"{epoch+1}/{epochs}| QAT training is in progress")
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            # Perform a forward pass through the model
            outputs = model(x)

            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            
            # Accumulate the running loss value
            running_loss += loss.item()
             # If labels are one-hot/soft (2D), target the index with the highest weight
            if y.ndim > 1:
                targets = torch.argmax(y, dim=1)
            else:
                targets = y

            
            # Determine the predicted class for accuracy calculation
            _, predicted = torch.max(outputs, 1)
            
            # Update the total count of processed samples
            total += y.size(0)
            
            # Update the count of correctly predicted samples
            correct += (predicted == targets).sum().item()
            
        # Calculate the average loss over all training batches
        avg_epoch_loss = running_loss / len(loader)
        
        # Calculate the final training accuracy for the epoch
        train_accuracy = 100 * correct / total
        val_acc=qat_evaluate(model,device,val_loader)
        print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {avg_epoch_loss:.4f}, Accuracy: {train_accuracy:.2f}%')
        print(f"Validation Accuracy {epoch + 1}/{epochs}: {val_acc*100:.2f}%")
    return model


def qat_quantization(model):
     #model have been set to eval mode here
    exported_model=export_model_for_quantize(model)
    #convert it back to train for QAT
    # torchao.quantization.pt2e.move_exported_model_to_train(exported_model)
    
    # quantization-aware training
    # Use Backend Quantizer for X86 CPU
    quantizer = X86InductorQuantizer()
    quantizer.set_global(xiq.get_default_x86_inductor_quantization_config(is_qat=True))
    prepared_model = prepare_qat_pt2e(exported_model, quantizer)

    return prepared_model


In [186]:
#prepare model for QAT
prepared_QAT=qat_quantization(qat_model_to_quantize)

In [187]:
#call train function 
trained_QAT=qat_train(prepared_QAT, train_loader,val_loader=val_loader,epochs=5, lr=1e-5,device=device)

1/5| QAT training is in progress:   0%|          | 0/313 [00:00<?, ?it/s]

Validation is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/5, Training Loss: 0.7254, Accuracy: 83.26%
Validation Accuracy 1/5: 95.09%


2/5| QAT training is in progress:   0%|          | 0/313 [00:00<?, ?it/s]

Validation is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

...

Validation is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 4/5, Training Loss: 0.6633, Accuracy: 85.50%
Validation Accuracy 4/5: 95.09%


5/5| QAT training is in progress:   0%|          | 0/313 [00:00<?, ?it/s]

Validation is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 5/5, Training Loss: 0.6659, Accuracy: 84.61%
Validation Accuracy 5/5: 95.12%


In [188]:
#convert to QAT quantized 
QAT_quantized=convert_pt2e(trained_QAT)
#set eval model and then move back to CPU for CPU inference and Compilation
QAT_quantized=torchao.quantization.pt2e.move_exported_model_to_eval(QAT_quantized.to("cpu"))

In [194]:
check_if_quantized(QAT_quantized)

Conv2d:  20/20 quantized
Linear:  1/1 quantized
Total:   21/21 quantized
>>> ALL ops quantized


In [195]:
clean_exported_qat=rebuild_clean_graph_and_save(QAT_quantized,"very_last_QAT_quantize")

W1018 08:22:50.684000 58 torch/export/pt2_archive/_package.py:686] Expect archive file to be a file ending in .pt2, or is a buffer. Instead got {../very_last_QAT_quantize.pt2e}


Output verification passed.
Original state entries: 105
Clean state entries: 84


In [196]:
compiled_qat_model=torch_inductor(clean_exported_qat,128)

Compiling and tuning kernels... (This takes a moment)


skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()


Compilation successful!


# Test on CPU inference

In [197]:
qat_exp5_v3_history=test_model(
    model=compiled_qat_model,
    test_dataloader=test_loader,
    num_classes=10,
    device=torch.device("cpu"),
    test_result_dir="./result/exp5_v3_last_qat"
        )

print(f"Test acc: {qat_exp5_v3_history["test_acc"]}")

test model is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()
skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()


Test acc: 92.59


In [198]:
results=measure_inference_mb_ms(model=compiled_qat_model,input_shape=(1,3,224,224),save_path="../last_qat_inference",experiment_name="exp5_v3",device=torch.device("cpu"))
print(f"Exp 5 V 3.0 size and inference time after QAT Quantization: {results}")

skipping cudagraphs due to cpp wrapper enabled
/usr/local/lib/python3.12/dist-packages/torch/_inductor/mkldnn_lowerings.py:764: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(w_zp_tensor, dtype=torch.int32), name=w_zp.get_name()


Average latency: 11.878 ms
State dict size: 11.28 MB
Exp 5 V 3.0 size and inference time after QAT Quantization: {'model_time_ms': 11.878050640098081, 'state_dict_size_mb': 11.276135, 'exported_file_size_mb': None}
